# Au+Au 200 GeV, 0–10 %: does the O+O chain's energy loss describe central Au+Au?

The calibration test of the tuning page ("A3 and the proposed runs"; `auau_bulk_phobos.ipynb`, "Proposed
runs"). The O+O physics-run chain (`OO_MCGlauber_MUSIC_isr_rhictune.xml`) with Au + Au, b < 4.7 fm (0–10 %),
MUSIC grid ±13 fm, hard partons |y| < 1 (either one), jet in the medium:

| Run | Samples | Energy loss | Hadronization |
| --- | --- | --- | --- |
| B1 | `auau_med_w*` | current: Matter type 1 (2204.01163's Type 2), α_s 0.25, Lbt 0.25 | iColored |
| B2 | `auau_t3_w*` | 2204.01163's default Type 3: type 5, α_s 0.3, qhatA 10, qhatB 100, Lbt α_s 0.3 | iColored |
| B1 / B2 Colorless | `rehad_colorless/auau_{med,t3}_w*` | B1's / B2's stored partons | Colorless (offline) |
| run C | `js_chain_t3` (GB10) | Type 3, JETSCAPE's jet chain (PythiaGun, Pythia ISR), one hydro leg | Colorless |
| T1–T3 | `hadtests/`, `auau_fake_w*` | B1 with `--matter-only` / closure `thermal` / `fake`, 18–40 GeV | iColored and Colorless |
| B1 / run C, no shift | `rehad_colorless_noshift/` | B1's / run C's stored partons | Colorless **without its close-pair shift** (offline, GB10) |
| run C, iColored | `icolored_runC/` | run C's stored partons (no ISR, no remnants) | iColored (offline, GB10, `rehadronize_icolored.py`): the hard partons' open lines on remnant-like **√s/6 beam ends** (`--beam-ends sqrts6`), the holes' lines by closure `fake` / `reconnect`; p+p: PP19's partons the same way. (First try: 1 GeV fake ends, superseded) |

B1 and B2: the same seeds, 10 p̂_T windows × 2k events each (complete 2026-11-26). All on the same medium (the
RHIC-tune 3dMCGlauber strings + 3D MUSIC, b < 4.7 fm).

**Data:**
- **STAR charged jets** (PRC 102, 054913 (2020), arXiv:2006.00582, Fig. 17, read from the vector figure:
  `refs/star_auau200_chjet/`): R_AA^Pythia, anti-k_T R = 0.2, 0.3, 0.4, |η_jet| < 1 − R, leading charged
  track > 5 GeV/c **on the Au+Au jets only: "No p_T,lead cut is imposed on this reference jet population"**
  (Sec. X B). **The p+p reference is PYTHIA (Perugia 2012, PARP(90) 0.213), not a measurement.** At low
  jet p_T the cut removes jets that the reference keeps, which pushes R_AA down: the "biased" points.
  STAR calls the range "~unbiased" above 15 GeV/c (R = 0.2) and 17 GeV/c (R = 0.3, 0.4).
- **JETSCAPE's own Type 3 calculation** (2204.01163 Fig. 14, read from the vector figure:
  `refs/jetscape_2204_fig14/`): the published default (α_s 0.3, Q_sw 2 GeV, τ₀ 0.6) in JETSCAPE's chain
  (TRENTo + free streaming + VISHNU, PP19 p+p, Colorless), the same STAR and PHENIX observables, no retuning
  at RHIC. Purple steps in Figs. 1 and 2.
- **PHENIX π⁰** (PRC 87, 034911 (2013), arXiv:1208.2254, Table VII: `refs/phenix_auau200_pi0/`): |y| < 0.35;
  global uncertainty 9.7 % (p+p normalisation) + 1.5 %.
- **STAR π±** (PRL 108, 072302 (2012), arXiv:1110.0579) and **STAR charged hadrons** (PRL 91, 172302 (2003),
  nucl-ex/0305015), both from HEPData with STAR's measured p+p reference: `refs/star_auau200_hadrons/`.

**Model:** R_AA(p_T) = Σ_AA w dN/dp_T / Σ_pp w dN/dp_T over the hard event's hadrons (one hard scattering
per Au+Au event; w = σ_window / N_window), against `pp_ref` (the same chain in p+p, phase 3 + 3e, with the
same |y| < 1 cut on the hardest partons). The medium's holes and the copies of their colour partners
(iColored hadronizes recoil + partner, the balance is recoil − hole) are subtracted inside the cone: full jets
with the full four-momentum, charged jets with the charged fraction 2/3 of the transverse momentum (≈ 3 GeV
per R = 0.4 jet in central Au+Au, against ≈ 0.1 GeV in O+O). Every jet R_AA is also shown without the
subtraction (thin dotted), to show its size. STAR's leading-track cut is applied as STAR does: to the Au+Au
jets only.
Pions (π⁺ + π⁻)/2 stand in for the π⁰ (decayed in these files). Hadron observables are not hole-subtracted
(`negative_hadrons.py`). The model's hadron R_AA is read from 10 GeV on (grey below): see Fig. 2. Centrality is by b (geometric 0–10 %), the data's by multiplicity.
**Model results, not vetted.**

> **Update (2026-11-27): superseded by `colorless_shift_study.ipynb`.** The samples here (B1, B2, run C, 20k events each) were the first look. The high-statistics study (run 1, JETSCAPE chain, and run 1′, our ISR chain with hydro reuse, 78k events each, 10 fragmentations per event, extended p+p references) is in `colorless_shift_study.ipynb` (js-contrib `280ddf7`): the close-pair shift result holds (without it R_AA is 4–5× higher for hadrons, 2.5–3.3× for jets), with the parton-level reference, the holes' closure with the real remnants and the effective number of events per bin (n_eff). Several single-bin values below (π⁰ bins, run C's hadrons at 10–25 GeV) rest on few events from the low p̂_T windows; use that notebook's numbers. Model results, not vetted.

In [ ]:
import json, subprocess, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
FAM = {"auau_med": ("B1: Type 2, α_s 0.25 (current)", "C3", "o"), "auau_t3": ("B2: Type 3, α_s 0.3", "C0", "s"),
       "auau_med_colorless": ("B1's partons, Colorless (vs pp_ref Colorless)", "C2", "^"),
       "auau_t3_colorless": ("B2's partons, Colorless (vs pp_ref Colorless)", "C8", "v"),
       "js_chain_t3": ("run C: JETSCAPE chain on our medium (vs PP19)", "C5", "D"),
       "auau_med_colorless_noshift": ("B1's partons, Colorless without the close-pair shift", "C2", "P"),
       "js_chain_t3_noshift": ("run C's partons, Colorless without the close-pair shift", "C5", "X"),
       "js_chain_t3_icolored_fake": ("run C's partons, iColored, closure fake (vs PP19 iColored)", "C1", "h"),
       "js_chain_t3_icolored_reconnect": ("run C's partons, iColored, closure reconnect (vs PP19 iColored)", "C6", "H"),
       "js_chain_t3_icolored_beamends_fake": ("run C's partons, iColored, √s/6 beam ends, holes fake", "C1", "p"),
       "js_chain_t3_icolored_beamends_reconnect": ("run C's partons, iColored, √s/6 beam ends, holes reconnect", "C6", "*")}
# the samples' files: oo_bias.py families by default; globs for the others (auau_raa.py --auau-glob / --pp-glob)
RC = "/data/AuAu-RHIC_phys/rehad_colorless"
RN = "/data/AuAu-RHIC_phys/rehad_colorless_noshift"     # Colorless with <close_pair_shift>0 (GB10, 2026-11-26)
RI = "/data/AuAu-RHIC_phys/icolored_runC"               # iColored on stored partons (rehadronize_icolored.py, GB10)
ARGS = {"auau_med_colorless": ["--auau-glob", f"{RC}/auau_med_w*/*_hadrons_jet_frag.h5",
                               "--pp-glob", f"{RC}/pp_ref_w*/*_hadrons_jet_frag.h5"],
        "auau_t3_colorless": ["--auau-glob", f"{RC}/auau_t3_w*/*_hadrons_jet_frag.h5",
                              "--pp-glob", f"{RC}/pp_ref_w*/*_hadrons_jet_frag.h5"],
        "js_chain_t3": ["--auau-glob", "/data/AuAu-RHIC_phys/js_chain_t3/*_hadrons_jet_frag.h5",
                        "--pp-glob", "/data/OO-RHIC_pp_refs/pp19_icdecays/*_hadrons_jet_frag.h5"],
        "auau_med_colorless_noshift": ["--auau-glob", f"{RN}/auau_med_w*/*_hadrons_jet_frag.h5",
                                       "--pp-glob", f"{RN}/pp_ref_w*/*_hadrons_jet_frag.h5"],
        "js_chain_t3_noshift": ["--auau-glob", f"{RN}/js_chain_t3/*_hadrons_jet_frag.h5",
                                "--pp-glob", f"{RN}/pp19_icdecays/*_hadrons_jet_frag.h5"],
        **{f"js_chain_t3_icolored_{c}": ["--auau-glob", f"{RI}/js_chain_t3_{c}/*_hadrons_jet_frag.h5",
                                         "--pp-glob", f"{RI}/pp19_icdecays_fake/*_hadrons_jet_frag.h5"] for c in ("fake", "reconnect")},
        **{f"js_chain_t3_icolored_beamends_{c}": ["--auau-glob", f"{RI}/js_chain_t3_beamends_{c}/*_hadrons_jet_frag.h5",
                                                  "--pp-glob", f"{RI}/pp19_icdecays_beamends/*_hadrons_jet_frag.h5"] for c in ("fake", "reconnect")}}
PRIMARY = ("auau_med", "auau_t3")   # our chain (iColored, pp_ref): all variants in Figs. 1-2, Figs. 3 and 5
NOSHIFT = ("auau_med_colorless_noshift", "js_chain_t3_noshift",   # Fig. 8 and Fig. 7 only
           "js_chain_t3_icolored_fake", "js_chain_t3_icolored_reconnect",
           "js_chain_t3_icolored_beamends_fake", "js_chain_t3_icolored_beamends_reconnect")
SHOW = ["auau_med", "auau_t3", "auau_med_colorless", "auau_t3_colorless", "js_chain_t3", *NOSHIFT]
FAM = {k: v for k, v in FAM.items() if k in SHOW}
RES = {}
for fam in FAM:
    if fam in ARGS or list(Path("/data/AuAu-RHIC_phys").glob(f"{fam}_w*/*.json")):
        r = subprocess.run([sys.executable, "auau_raa.py", "--family", fam, "-j", "6", *ARGS.get(fam, [])],
                           capture_output=True, text=True)
        print(r.stdout[-2500:] if r.returncode == 0 else r.stderr[-1500:])
        p = Path(f"results/auau_raa_{fam}.json")
        if p.exists():
            RES[fam] = json.loads(p.read_text())
    else:
        print(f"{fam}: no finished jobs yet")
STAR = json.loads(Path("refs/star_auau200_chjet/star_chjet_raa.json").read_text())["centrality"]
PHX = json.loads(Path("refs/phenix_auau200_pi0/phenix_pi0_raa_0_10.json").read_text())["points"]
JS14 = json.loads(Path("refs/jetscape_2204_fig14/js_fig14.json").read_text())["panels"]   # JETSCAPE Type 3, Fig. 14
def js_step(ax_, key, **kw):
    q = JS14[key]
    e = [q[0]["lo"]] + [r["hi"] for r in q]; v = [r["value"] for r in q]
    ax_.stairs(v, e, baseline=None, color="C4", lw=1.8, **kw)
    ax_.errorbar([0.5 * (r["lo"] + r["hi"]) for r in q], v, [r["err_hi"] for r in q], fmt="none", color="C4", lw=1)
def model(fam, key):
    d = RES[fam]["raa"][key]; e = np.array(d["edges"])
    return 0.5 * (e[1:] + e[:-1]), 0.5 * np.diff(e), np.array(d["raa"], float), np.array(d["err"], float)
for fam, r in RES.items():
    print(f"{fam}: {r['auau_events']} Au+Au events in windows {r['auau_windows']}; pp_ref {r['pp_events_in_cut']} events in the cut")

## STAR charged-jet R_AA, 0–10 %

Points with boxes: STAR (filled: "unbiased", open: "biased"; magenta-edged box: correlated, grey: shape
uncertainty; T_AA and PYTHIA normalisation not shown). Model, solid: STAR's definition, the leading-track
cut on Au+Au only (`chjet_R*_star`); dashed: no cut in either; dotted: STAR's definition without the hole
subtraction.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6), sharey=True)
for j, R in enumerate((0.2, 0.3, 0.4)):
    for q in STAR["0-10"][f"R{R}"]:
        x, w = 0.5 * (q["lo"] + q["hi"]), 0.5 * (q["hi"] - q["lo"])
        for k, c, al in (("sys_shape", "0.75", 0.6), ("sys_corr", "m", 0.0)):
            if f"{k}_hi" in q:
                ax[j].add_patch(plt.Rectangle((q["lo"], q["value"] - q[f"{k}_lo"]), 2 * w, q[f"{k}_lo"] + q[f"{k}_hi"],
                                              facecolor=c if al else "none", edgecolor=c, alpha=al if al else 1, lw=1))
        ax[j].plot(x, q["value"], "*", ms=10, color="navy", mfc="navy" if q["kind"] == "unbiased" else "white")
    ax[j].plot([], [], "*", ms=10, color="navy", label="STAR 0-10 % (open: biased)")
    js_step(ax[j], f"chjet_R{R}", label="JETSCAPE 2204.01163 Fig. 14: Type 3, α_s 0.3 (PP19 p+p)")
    for fam, (lab, c, mk) in FAM.items():
        if fam not in RES or fam in NOSHIFT:
            continue
        x, xe, r, e = model(fam, f"chjet_R{R}_star")
        ax[j].errorbar(x, r, e, xe, fmt=mk + "-", ms=4, color=c, label=f"{lab}, p_T,lead > 5 on Au+Au only (STAR)")
        if fam not in PRIMARY:
            continue
        x, xe, r, e = model(fam, f"chjet_R{R}")
        ax[j].plot(x, r, mk + "--", ms=3, mfc="none", color=c, alpha=0.6, label=f"{lab}, no lead cut")
        x, xe, r, e = model(fam, f"chjet_R{R}_star_raw")
        ax[j].plot(x, r, ":", lw=1.2, color=c, label=f"{lab}, STAR's definition, holes not subtracted")
    ax[j].set_title(f"charged jets, R = {R}, |η| < {1 - R:.1f}", fontsize=10); ax[j].set_xlabel(r"$p_{T,jet}^{ch}$ [GeV]")
    ax[j].set_ylim(0, 1.2); ax[j].axhline(1, color="k", lw=0.8); ax[j].set_xlim(5, 32)
ax[0].set_ylabel(r"$R_{AA}$"); ax[0].legend(fontsize=7)
fig.suptitle("Figure 1. Au+Au 0-10 %: charged-jet R_AA, the O+O chain's energy loss against STAR (model, not vetted)", fontsize=11)
fig.tight_layout(); plt.show()
for fam in RES:
    for R in (0.2, 0.3, 0.4):
        x, xe, r, e = model(fam, f"chjet_R{R}_star")
        xr, _, rraw, _ = model(fam, f"chjet_R{R}_star_raw")
        d5 = RES[fam]["raa"][f"chjet_R{R}_lead5"]; d0 = RES[fam]["raa"][f"chjet_R{R}"]
        print(f"{fam} R={R}: p+p fraction of jets with a 5 GeV leading hadron (STAR's PYTHIA 5/0 > 0.9 above 13 / 17 GeV for R = 0.2 / 0.4) "
              + " ".join(f"{xx:.0f}:{a5 / a0:.2f}" for xx, a5, a0 in zip(x, d5["pp"], d0["pp"]) if a0 > 0))
        print(f"{fam} R={R}: R_AA without / with the hole subtraction " + " ".join(f"{xx:.0f}:{a_ / b_:.2f}" for xx, a_, b_ in zip(x, rraw, r) if np.isfinite(a_) and b_ > 0))
        dat = {round(0.5 * (q["lo"] + q["hi"]), 1): q["value"] for q in STAR["0-10"][f"R{R}"]}
        print(f"{fam} R={R}: model/STAR " + " ".join(f"{xx:.0f}:{rr / dat[round(xx, 1)]:.2f}" for xx, rr in zip(x, r) if round(xx, 1) in dat and np.isfinite(rr)))

## Hadron R_AA, 0–10 %: PHENIX π⁰, STAR π±, STAR charged hadrons

- **PHENIX π⁰** (0–10 %, |y| < 0.35): statistical bars, grey boxes type-B systematics; 9.7 % normalisation not shown.
- **STAR π⁺ + π⁻** (PRL 108, 072302 (2012), arXiv:1110.0579, HEPData ins930463; **0–12 %**, |y| < 0.5; measured
  p+p reference): bars statistical, boxes systematic; ≈ 12 % p+p normalisation not shown.
- **STAR (h⁺ + h⁻)/2** (PRL 91, 172302 (2003), nucl-ex/0305015, HEPData ins619063; |η| < 0.5; measured p+p):
  **0–5 %** (filled) and **10–20 %** (open), which bracket our geometric 0–10 %; bars are stat + sys; the N_bin
  normalisation (±16 %) is not shown. `refs/star_auau200_hadrons/fetch_star_hadron_raa.py`.

Model in each experiment's acceptance: pions (π⁺ + π⁻)/2 at |y| < 0.5 (STAR's; solid) and |η| < 0.35 (PHENIX's;
open, faint), charged hadrons |η| < 0.5. **Grey below 10 GeV: the model is statistics-limited there**: hadrons
at 5–10 GeV come mostly from the lowest p̂_T windows (5–11 GeV), where only ≈ 10–50 of 2000 events per window
carry such a hadron (in Au+Au the 5–7 GeV window alone holds half of the 5–8 GeV yield, from 10 events). Medium
response (hadrons are not hole-subtracted) adds to it at the low end. Right panel: model only.

In [ ]:
SH = json.loads(Path("refs/star_auau200_hadrons/star_hadron_raa.json").read_text())
def integ(fam, key, lo, hi):
    d = RES[fam]["raa"][key]; e = np.array(d["edges"]); a = np.array(d["auau"]); p = np.array(d["pp"]); er = np.array(d["err"])
    s = (e[:-1] >= lo - 1e-9) & (e[1:] <= hi + 1e-9)
    return a[s].sum() / p[s].sum(), np.sqrt(((er * p)[s] ** 2).sum()) / p[s].sum()
fig, ax = plt.subplots(1, 3, figsize=(19, 4.6))
for q in PHX:
    ax[0].add_patch(plt.Rectangle((q["lo"], q["value"] - q["syst_B"]), q["hi"] - q["lo"], 2 * q["syst_B"], color="0.6", alpha=0.35, lw=0))
ax[0].errorbar([q["pt"] for q in PHX], [q["value"] for q in PHX], [q["stat"] for q in PHX], fmt="o", ms=4, color="k", label="PHENIX π⁰ 0-10 %, |y| < 0.35")
for q in SH["pion"]["points"]:
    if q["hi"] <= 4.5:
        continue
    ax[0].add_patch(plt.Rectangle((q["lo"], q["value"] - q["syst"]), q["hi"] - q["lo"], 2 * q["syst"], facecolor="none", edgecolor="navy", lw=0.8))
    ax[0].errorbar(0.5 * (q["lo"] + q["hi"]), q["value"], q["stat"], fmt="*", ms=9, color="navy")
ax[0].plot([], [], "*", ms=9, color="navy", label="STAR π± 0-12 %, |y| < 0.5")
js_step(ax[0], "pion", label="JETSCAPE 2204.01163 Fig. 14: Type 3, (π⁺+π⁻)/2 |y| < 0.35")
for c_, (ps, mfc) in {"0-5%": ("navy", "navy"), "10-20%": ("navy", "white")}.items():
    pts = [q for q in SH["hadron"]["centrality"][c_] if q["hi"] > 4.5]
    ax[1].errorbar([0.5 * (q["lo"] + q["hi"]) for q in pts], [q["value"] for q in pts], [q["stat_syst"] for q in pts],
                   [0.5 * (q["hi"] - q["lo"]) for q in pts], fmt="D", ms=5, color=ps, mfc=mfc, label=f"STAR (h⁺+h⁻)/2 {c_}, |η| < 0.5")
for fam, (lab, c, mk) in FAM.items():
    if fam not in RES or fam in NOSHIFT:
        continue
    x, xe, r, e = model(fam, "pion_star")
    ax[0].errorbar(x, r, e, xe, fmt=mk, ms=4, color=c, label=f"{lab}: π±, |y| < 0.5")
    if fam in PRIMARY:
        x, xe, r, e = model(fam, "pion")
        ax[0].errorbar(x, r, e, xe, fmt=mk, ms=3, mfc="none", color=c, alpha=0.35, label=f"{lab}: π±, |η| < 0.35")
    x, xe, r, e = model(fam, "had_star")
    ax[1].errorbar(x, r, e, xe, fmt=mk + "-", ms=4, color=c, label=f"{lab}: (h⁺+h⁻)/2, |η| < 0.5")
    x, xe, r, e = model(fam, "had")
    ax[2].errorbar(x, r, e, xe, fmt=mk + "-", ms=4, color=c, label=f"{lab}: charged hadrons |η| < 1")
    x, xe, r, e = model(fam, "fjet_R0.4")
    ax[2].errorbar(x, r, e, xe, fmt=mk + "--", ms=4, mfc="none", color=c, label=f"{lab}: full jets R = 0.4, |η| < 0.5")
    if fam in PRIMARY:
        x, xe, r, e = model(fam, "fjet_R0.4_raw")
        ax[2].plot(x, r, ":", lw=1.2, color=c, label=f"{lab}: full jets, holes not subtracted")
for a_ in ax:
    a_.axhline(1, color="k", lw=0.8); a_.set_ylim(0, 1.6); a_.set_xlabel(r"$p_T$ [GeV]"); a_.legend(fontsize=7, loc="upper right")
for a_ in ax[:2]:
    a_.axvspan(0, 10, color="0.92", zorder=0)
ax[0].set_xlim(4.5, 20.5); ax[1].set_xlim(4.5, 15.5); ax[2].set_xlim(4.5, 50)
ax[0].set_ylabel(r"$R_{AA}$")
ax[0].set_title("pions: PHENIX π⁰, STAR π±; grey: model statistics-limited", fontsize=10)
ax[1].set_title("charged hadrons: STAR 0-5 % and 10-20 % bracket 0-10 %", fontsize=10)
ax[2].set_title("model only: charged hadrons and full jets", fontsize=10)
fig.suptitle("Figure 2. Au+Au 0-10 %: hadron R_AA against PHENIX and STAR, and the model's full-jet R_AA (model, not vetted)", fontsize=11)
fig.tight_layout(); plt.show()
st_pi = [q for q in SH["pion"]["points"] if q["lo"] >= 10]
st_h = {c_: [q for q in SH["hadron"]["centrality"][c_] if q["lo"] >= 8] for c_ in ("0-5%", "10-20%")}
print("data 10-15 GeV: STAR pi+- " + " ".join(f"{q['lo']:g}-{q['hi']:g}:{q['value']:.2f}" for q in st_pi)
      + " | PHENIX pi0 " + " ".join(f"{q['lo']:g}-{q['hi']:g}:{q['value']:.2f}" for q in PHX if 10 <= q["lo"] < 15)
      + " | STAR h 8-10 GeV " + "; ".join(f"{c_} " + " ".join(f"{q['value']:.2f}" for q in v) for c_, v in st_h.items()))
for fam in RES:
    for key, lo, hi in (("pion_star", 10, 15), ("pion", 10, 20), ("had_star", 10, 15), ("had", 10, 25)):
        r_, e_ = integ(fam, key, lo, hi)
        print(f"{fam}: {key} {lo}-{hi} GeV (integrated): R_AA {r_:.2f} ± {e_:.2f}")

## The p+p reference: data, Pythia 8 and JETSCAPE PP19

**What `pp_ref` is.** The full chain in p+p (`pp200_isr_rhictune.xml`, b < 3 fm): 3dMCGlauber strings, MUSIC
and iSS where the event has a string (29 % of the jet events; 71 % have no medium at all), PythiaIsrGun +
iMATTER, the hard-momentum subtraction and string veto as in O+O, iColored. **The jet stays in vacuum**
(`vacuum_jet 1`: Matter `in_vac 1` showers to Q0 = 2 GeV, no LBT, no recoils), so the hydro does not touch
the jet observables; the bulk is in a separate file and not used here. The Au+Au medium arm showers the
vacuum part to QS = 0.9 GeV instead (A+A scheme); that changes jets by < 1 % and leading hadrons by at most
a few % (tuning page, "Phase 4"). The hard tune fitted α_s(M_Z) = 0.153 to STAR's p+p full jets.

**Comparison** (`pp_ref_check.py`, the same code for all samples, hard event only): `pp_ref` (70k events),
Pythia 8.312 default (Monash) and STAR's Detroit tune (`refs/pythia_ref.py`, 160k events each, the same
p̂_T windows, /data/OO-RHIC_pp_refs) and JETSCAPE's PP19 (arXiv:1910.05481, the p+p baseline of 2204.01163:
Pythia's ISR and MPI, MATTER in vacuum to Q0 = 1 GeV, Colorless; `refs/pp19_200.xml`,
`refs/hadronize_pp19_icdecays.xml`, 160k events), all **with iColored's decays**: iColored (X-SCAPE
`iColoredHadronization.cc`, no `<weak_decays>` in our XML) sets `HadronLevel:Decay on`,
`ParticleDecays:limitTau0 on`, `tau0Max = 10 mm`, so π⁰, η and all resonances decay and K⁰_S, Λ, Σ±, Ξ, Ω do
not. That is the data's "primary" definition: STAR subtracts the weak-decay feed-down (K⁰_S → π⁺π⁻) from its
π± and corrects its h± for Λ, K⁰_S, Σ daughters; its charged jets use primary tracks (DCA < 1 cm). The Pythia
samples get the same four lines (`--line`; `*_icdecays`), checked by their species content (K⁰_S 2.10 %,
Λ 0.69 %, e± 0.62 %, no π⁰/η, against `pp_ref`'s 2.14 / 0.74 / 0.62 %). `pythia_ref.py`'s own default (all
decays except π⁰, η) adds the K⁰_S and Λ daughters to the charged jets: + 5–18 % at R = 0.4. Data: STAR π± and (h⁺+h⁻)/2 p+p (the references of the STAR R_AA above;
NSD yields × 30 mb), PHENIX π⁰ (`data/data.json`), STAR full jets R = 0.5 (`pp_jet_spectra.py`).
**STAR's charged-jet R_AA uses PYTHIA 6 Perugia 2012 with PARP(90) = 0.213 as p+p** (2006.00582 Sec. IX):
not available here; Monash and Detroit stand in for it. Pythia's jets include its underlying event;
"UE-subtracted" removes the mean charged p_T in two perpendicular cones (≈ 0.2 GeV per R = 0.4 jet in Pythia,
0.1 GeV in `pp_ref`). The p̂_T windows start at 5 GeV, so hadrons are compared from 6 GeV.

In [ ]:
if not Path("results/pp_ref_check.json").exists():
    subprocess.run([sys.executable, "pp_ref_check.py", "-j", "6"], check=True, capture_output=True)
PR = json.loads(Path("results/pp_ref_check.json").read_text())["samples"]
DD = json.loads(Path("data/data.json").read_text())
PJ = json.loads(Path("results/pp_jet_spectra.json").read_text())
PPS = {"pp_ref": ("pp_ref (our chain)", "k", "o"), "pythia8_default_ic": ("Pythia 8 Monash", "C1", "s"),
       "pythia8_detroit_ic": ("Pythia 8 Detroit", "C2", "^"),
       "pp19_ic": ("JETSCAPE PP19", "C4", "D")}
def pdata(k):
    if k == "pi_star":
        return {(q["lo"], q["hi"]): (q["value"] * 30, np.hypot(q["stat"], q["syst"]) * 30) for q in SH["pp_pion"]["points"]}
    if k == "h_star":
        return {(q["lo"], q["hi"]): (q["value"] * 30, q["stat_syst"] * 30) for q in SH["pp_hadron"]["points"]}
    P_ = DD["phenix_pi0"]
    return {tuple(e): (v, np.hypot(a, b)) for e, v, a, b in zip(P_["edges"], P_["value"], P_["stat"], P_["sys"])}
def ratio_to_data(n, k, lo=6):
    s = PR[n]["spectra"][k]; e = s["edges"]; dd = pdata(k); x, xe, r, er, de = [], [], [], [], []
    for i in range(len(e) - 1):
        if e[i] < lo or (e[i], e[i + 1]) not in dd:
            continue
        v, dv = dd[(e[i], e[i + 1])]
        x.append(0.5 * (e[i] + e[i + 1])); xe.append(0.5 * (e[i + 1] - e[i])); r.append(s["value"][i] / v); er.append(s["err"][i] / v); de.append(dv / v)
    return map(np.array, (x, xe, r, er, de))
fig, ax = plt.subplots(1, 4, figsize=(21, 4.4))
for j_, (k, tit, lo) in enumerate((("pi_star", "π±, |y| < 0.5: model / STAR p+p", 6), ("pi_phenix", "π (model) / PHENIX π⁰ p+p, |y| < 0.35", 6),
                                    ("h_star", "(h⁺+h⁻)/2, |η| < 0.5: model / STAR p+p", 6))):
    for n, (lab, c, mk) in PPS.items():
        x, xe, r, er, de = ratio_to_data(n, k, lo)
        ax[j_].errorbar(x + {"pp_ref": 0, "pythia8_default_ic": 0.12, "pythia8_detroit_ic": -0.12, "pp19_ic": 0.24}[n], r, er, fmt=mk, ms=4, color=c, label=lab)
    for xx, xw, d_ in zip(x, xe, de):
        ax[j_].add_patch(plt.Rectangle((xx - xw, 1 - d_), 2 * xw, 2 * d_, color="0.85", lw=0, zorder=0))
    ax[j_].set_title(tit, fontsize=10); ax[j_].set_ylim(0, 2.5); ax[j_].axhline(1, color="k", lw=0.8); ax[j_].set_xlabel(r"$p_T$ [GeV]")
ax[0].set_ylabel("model / data"); ax[0].legend(fontsize=7)
for Rj, ls in ((0.2, ":"), (0.3, "--"), (0.4, "-")):
    e = np.array(PR["pp_ref"]["spectra"][f"chjet_R{Rj}_uesub"]["edges"]); xc = 0.5 * (e[1:] + e[:-1])
    a_ = np.array(PR["pp_ref"]["spectra"][f"chjet_R{Rj}_uesub"]["value"])
    for n, c in (("pythia8_detroit_ic", "C2"), ("pythia8_default_ic", "C1"), ("pp19_ic", "C4")):
        ax[3].plot(xc, a_ / np.array(PR[n]["spectra"][f"chjet_R{Rj}_uesub"]["value"]), ls, color=c, marker="o", ms=3,
                   label=f"R = {Rj}: pp_ref / {PPS[n][0].split()[-1]}")
st = DD["star_jets_eta0"]; ed = np.array(st["edges"]); m_ = np.array(PJ["star_eta0"]["value"]); dv = np.array(st["value"])
ax[3].plot(ed.mean(1), m_ / dv, "k*", ms=8, label="full jets R = 0.5, |η| < 0.5: pp_ref / STAR p+p")
ax[3].set_ylim(0, 1.6); ax[3].set_xlim(7, 32); ax[3].axhline(1, color="k", lw=0.8); ax[3].legend(fontsize=6.5, loc="lower left")
ax[3].set_title("charged jets (UE-subtracted): pp_ref / Pythia 8, PP19", fontsize=10); ax[3].set_xlabel(r"$p_{T,jet}$ [GeV]")
fig.suptitle("Figure 4. The p+p reference: pp_ref, Pythia 8 and JETSCAPE PP19 against p+p data, and pp_ref's charged jets against them (model, not vetted)", fontsize=11)
fig.tight_layout(); plt.show()
for k, lo, hi in (("pi_star", 6, 15), ("pi_phenix", 6, 20), ("h_star", 6, 10)):
    row = []
    for n in PPS:
        x, xe, r, er, de = ratio_to_data(n, k)
        s_ = (x - xe >= lo) & (x + xe <= hi); w_ = 1 / er[s_] ** 2
        row.append(f"{n} {np.sum(r[s_] * w_) / w_.sum():.2f} ± {w_.sum() ** -0.5:.2f}")
    print(f"{k} {lo}-{hi} GeV, model/data (weighted): " + " | ".join(row))
for Rj in (0.2, 0.3, 0.4):
    a_ = np.array(PR["pp_ref"]["spectra"][f"chjet_R{Rj}_uesub"]["value"])
    print(f"charged jets R = {Rj} (UE-subtracted), pp_ref / Detroit " + " ".join(f"{v:.2f}" for v in a_ / np.array(PR["pythia8_detroit_ic"]["spectra"][f"chjet_R{Rj}_uesub"]["value"]))
          + " | / Monash " + " ".join(f"{v:.2f}" for v in a_ / np.array(PR["pythia8_default_ic"]["spectra"][f"chjet_R{Rj}_uesub"]["value"]))
          + " | / PP19 " + " ".join(f"{v:.2f}" for v in a_ / np.array(PR["pp19_ic"]["spectra"][f"chjet_R{Rj}_uesub"]["value"])))

### What the reference does to the jet R_AA

STAR divides by PYTHIA, we divide by `pp_ref`. Figure 5 divides the model's Au+Au charged jets (STAR's
definition: leading-track cut on Au+Au only) by each p+p sample: solid `pp_ref` (the R_AA of Fig. 1), dashed
Pythia 8 Detroit, dotted Monash, dash-dotted JETSCAPE PP19 (2204.01163's p+p baseline), all without the leading-track cut and with their own underlying event, as
STAR's PYTHIA reference. This emulates STAR's construction with a PYTHIA 8 stand-in for PYTHIA 6.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.4), sharey=True)
for j, Rj in enumerate((0.2, 0.3, 0.4)):
    for q in STAR["0-10"][f"R{Rj}"]:
        ax[j].add_patch(plt.Rectangle((q["lo"], q["value"] - q.get("sys_corr_lo", 0)), q["hi"] - q["lo"],
                                      q.get("sys_corr_lo", 0) + q.get("sys_corr_hi", 0), facecolor="none", edgecolor="m", lw=1))
        ax[j].plot(0.5 * (q["lo"] + q["hi"]), q["value"], "*", ms=10, color="navy", mfc="navy" if q["kind"] == "unbiased" else "white")
    for fam, (lab, c, mk) in FAM.items():
        if fam not in PRIMARY or fam not in RES:          # our chain's Au+Au over the p+p references
            continue
        d = RES[fam]["raa"][f"chjet_R{Rj}_lead5"]; e = np.array(d["edges"]); xc = 0.5 * (e[1:] + e[:-1])
        A_ = np.array(d["auau"]) / np.diff(e) / (2 * (1 - Rj))
        for n, ls, cc in (("pp_ref", "-", c), ("pythia8_detroit_ic", "--", c), ("pythia8_default_ic", ":", c), ("pp19_ic", "-.", "C4")):
            pp_ = np.array(PR[n]["spectra"][f"chjet_R{Rj}"]["value"])
            ax[j].plot(xc, A_ / pp_, ls, color=cc, marker=mk if n == "pp_ref" else None, ms=4, label=f"{lab} / {PPS[n][0]}")
        print(f"{fam} R = {Rj}: Au+Au / pp_ref " + " ".join(f"{v:.2f}" for v in A_ / np.array(PR['pp_ref']['spectra'][f'chjet_R{Rj}']['value']))
              + " | / Detroit " + " ".join(f"{v:.2f}" for v in A_ / np.array(PR['pythia8_detroit_ic']['spectra'][f'chjet_R{Rj}']['value']))
              + " | / Monash " + " ".join(f"{v:.2f}" for v in A_ / np.array(PR['pythia8_default_ic']['spectra'][f'chjet_R{Rj}']['value']))
              + " | / PP19 " + " ".join(f"{v:.2f}" for v in A_ / np.array(PR['pp19_ic']['spectra'][f'chjet_R{Rj}']['value'])))
    ax[j].set_title(f"charged jets R = {Rj}, STAR's definition", fontsize=10); ax[j].set_xlabel(r"$p_{T,jet}^{ch}$ [GeV]")
    ax[j].set_ylim(0, 1.0); ax[j].set_xlim(7, 31)
ax[0].plot([], [], "*", color="navy", label="STAR 0-10 % (open: biased)"); ax[0].legend(fontsize=7); ax[0].set_ylabel(r"$R_{AA}$")
fig.suptitle("Figure 5. The model's charged-jet R_AA with four p+p references: pp_ref, Pythia 8 (as STAR) and JETSCAPE PP19 (model, not vetted)", fontsize=11)
fig.tight_layout(); plt.show()

## Where B1's under-quenching comes from: medium, energy loss or hadronization

Three runs on the **same medium** (the RHIC-tune 3dMCGlauber strings + 3D MUSIC, Au+Au b < 4.7 fm) separate
the pieces of the chain (2026-11-26):

| Sample | Hard process | Energy loss | Hadronization (holes) | p+p reference |
| --- | --- | --- | --- | --- |
| **B1** | PythiaIsrGun + iMATTER ISR, our RHIC hard tune | Type 2, α_s 0.25 | iColored, `colour_closure reconnect` (holes + partner copies subtracted at parton level, charged × 2/3) | `pp_ref` (iColored) |
| **B1's partons, Colorless** | the same events | the same final partons | Colorless on B1's stored `partons/` (`prod_OO_isr/rehadronize_colorless.py`; holes → pstat −1 hadrons, subtracted at hadron level) | `pp_ref`'s partons, Colorless |
| **run C** | PythiaGun, Pythia's ISR/MPI (PP19's hard process) | Type 3, α_s 0.3 (2204.01163's default) | Colorless (pstat −1 hadrons subtracted) | PP19 (Colorless) |
| (B2) | as B1 | Type 3, α_s 0.3 | as B1 | `pp_ref` |
| **T1** (Fig. 6c) | as B1 | **MATTER only** (no LBT, recoils, holes; so no closure) | iColored, and the same partons with Colorless | `pp_ref` both ways |
| **T2** (Fig. 6c) | as B1 | as B1 | iColored with `--colour-closure thermal`, and Colorless on the same partons | `pp_ref` both ways |
| **T3** (Fig. 6c) | as B1 | as B1 | iColored with `--colour-closure fake`, and Colorless on the same partons | `pp_ref` both ways |

Run C ran on the GB10 with `prod_AuAu_0_10_jet/run_prod_jet_single.py` (one hydro leg, no liquefier, reuse
20: 20k jets on 1,000 hydros, seeds 941001–941100, X-SCAPE 299fd3f as the 3090 image), with
`refs/hadronize_pp19_icdecays.xml` (Colorless with iColored's decays). All samples use the same decays.

**Logic.** JETSCAPE's Fig. 14 (their chain, their medium) vs run C (their chain, our medium) isolates the
medium. Run C vs B1 differs in the hard process, the q̂ setting and the hadronization; B1's partons
re-hadronized with Colorless isolate the **hadronization** exactly (same events, same partons, same
energy loss). The parton-level counts (Fig. 6) tell where the yields part: energy given to the medium,
final partons, hadrons per parton. Parton counts compare only within one shower scheme (the A+A scheme of
all Au+Au samples showers to QS = 0.9 GeV; `pp_ref` to Q0 = 2 GeV; PP19 to Q0 = 1 GeV): compare the Au+Au
samples with each other, and hadrons-per-parton within a sample type.

In [ ]:
if not Path("results/chain_diag.json").exists():
    subprocess.run([sys.executable, "chain_diag.py", "-j", "6"], check=True, capture_output=True)
CD = json.loads(Path("results/chain_diag.json").read_text())
STY = {"B1 (our chain, Type 2, iColored)": ("C3", "o", "-"), "B1 rehadronized with Colorless": ("C2", "^", "-"),
       "B2 (our chain, Type 3, iColored)": ("C0", "s", "-"), "B2 rehadronized with Colorless": ("C8", "v", "-"),
       "run C (JETSCAPE chain, Type 3, Colorless)": ("C5", "D", "-"),
       "T1 (our chain, MATTER only, iColored)": ("C1", "v", "--"), "T1 rehadronized with Colorless": ("C9", "v", "--"),
       "pp_ref (iColored, vacuum)": ("C3", "o", ":"), "pp_ref rehadronized with Colorless": ("C2", "^", ":"),
       "PP19 (Colorless, vacuum)": ("C5", "D", ":")}
fig, ax = plt.subplots(1, 4, figsize=(21, 4.4))
for name, d in CD.items():
    if name.startswith(("T1", "T2", "T3")):               # the hadronization tests: Fig. 6c
        continue
    c, mk, ls = STY.get(name, ("0.5", "x", "-"))
    w = [k for k in d["windows"] if float(k.split("-")[0]) >= 9]
    xc = [np.mean([float(v) for v in k.split("-")]) for k in w]
    get = lambda q: [d["windows"][k][q] for k in w]
    aa = ls != ":"
    if aa:
        ax[0].plot(xc, get("e_medium"), mk + ls, color=c, label=name)
        ax[1].plot(xc, get("par_pt15"), mk + ls, color=c, label=name)
    ax[2].plot(xc, get("had_pt12"), mk + ls, color=c, mfc=None if aa else "none", label=name)
    ok = np.array(get("par_pt15")) > 0.05                 # hadrons per parton only where partons > 15 GeV exist
    ax[3].plot(np.array(xc)[ok], np.array(get("had_per_par"))[ok], mk + ls, color=c, mfc=None if aa else "none", label=name)
ax[0].set_title("energy given to the medium: Σ E(recoils) − Σ E(holes) [GeV/event]", fontsize=9)
ax[1].set_title("final partons p_T > 15 GeV, |η| < 1 [/event] (A+A scheme)", fontsize=9)
ax[2].set_title("charged hadrons p_T > 12 GeV, |η| < 1 [/event]", fontsize=9)
ax[3].set_title("hadrons (p_T > 12) per parton (p_T > 15)", fontsize=9)
for a_ in ax:
    a_.set_xlabel("p̂_T window centre [GeV]"); a_.set_ylim(0, None)
ax[3].legend(fontsize=6.5); ax[1].set_yscale("linear")
fig.suptitle("Figure 6. Same medium, three chains: where between energy loss, partons and hadrons B1 and run C part (model, not vetted)", fontsize=11)
fig.tight_layout(); plt.show()
for w in ("11-14", "23-30", "40-55"):
    print(f"window {w} GeV:  e_medium | holes | partons>15 | hadrons>12 | hadrons/parton")
    for name, d in CD.items():
        q = d["windows"].get(w)
        if q:
            print(f"   {name:44s} {q['e_medium']:6.2f} | {q['holes']:6.1f} | {q['par_pt15']:.3f} | {q['had_pt12']:.3f} | {q['had_per_par']:.3f}")

### Fig. 6c: iColored vs Colorless on the same partons (the hadronization tests)

One number per p̂_T window: the charged hadrons above 8 (left) and 12 GeV (right) per event from **iColored**
divided by those from **Colorless hadronizing the same stored partons** (`prod_OO_isr/rehadronize_colorless.py`).
1 means the two hadronizations agree; > 1 is iColored's excess. Samples (all with B1's seeds and settings
except the one change; T1–T3: 500 events per window, 18–40 GeV):

- **vacuum** (`pp_ref`): no medium;
- **T1** (`--matter-only`, GB10): the medium modifies the MATTER shower, but there are no recoils, no holes
  and so no colour closure;
- **T2** (`--colour-closure thermal`, GB10) and **T3** (`--colour-closure fake`, 3090s): B1 with another way of
  closing the colour lines the holes leave open;
- **B1** (MATTER + LBT with recoils and holes, `--colour-closure reconnect`, the default).

Errors: from the hadron counts (Poisson, both samples independent): an over-estimate, since both use the same
partons.

In [ ]:
PAIRS6C = [("vacuum (pp_ref)", "pp_ref (iColored, vacuum)", "pp_ref rehadronized with Colorless", "k", "o"),
           ("T1: MATTER only", "T1 (our chain, MATTER only, iColored)", "T1 rehadronized with Colorless", "C1", "v"),
           ("T2: MATTER + LBT, closure thermal", "T2 (our chain, closure thermal, iColored)", "T2 rehadronized with Colorless", "C4", "<"),
           ("T3: MATTER + LBT, closure fake", "T3 (our chain, closure fake, iColored)", "T3 rehadronized with Colorless", "C9", ">"),
           ("B1: MATTER + LBT, closure reconnect", "B1 (our chain, Type 2, iColored)", "B1 rehadronized with Colorless", "C3", "s")]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4), sharey=True)
rows = []
for j, (thr, key) in enumerate(((8, "had_pt8"), (12, "had_pt12"))):
    for i, (lab, a_, b_, c, mk) in enumerate(PAIRS6C):
        if a_ not in CD or b_ not in CD:
            continue
        ws = [w for w in ("18-23", "23-30", "30-40") if w in CD[a_]["windows"] and w in CD[b_]["windows"]]
        xs, ys, es = [], [], []
        for w in ws:
            qa, qb = CD[a_]["windows"][w], CD[b_]["windows"][w]
            na, nb = qa[key] * qa["events"], qb[key] * qb["events"]
            if na > 0 and nb > 0:
                r = qa[key] / qb[key]
                xs.append(np.mean([float(v) for v in w.split("-")]) + (i - 2) * 0.35); ys.append(r); es.append(r * np.sqrt(1 / na + 1 / nb))
                rows.append((lab, thr, w, r, r * np.sqrt(1 / na + 1 / nb)))
        ax[j].errorbar(xs, ys, es, fmt=mk + "-", ms=7, color=c, label=lab)
    ax[j].axhline(1, color="k", lw=0.8); ax[j].set_xlabel("p̂_T window centre [GeV]")
    ax[j].set_title(f"charged hadrons p_T > {thr} GeV, |η| < 1: iColored / Colorless, same partons", fontsize=9)
ax[0].set_ylabel("iColored / Colorless"); ax[0].legend(fontsize=8); ax[0].set_ylim(0.5, 4.5)
fig.suptitle("Figure 6c. The hadronization tests: how much more high-p_T hadrons iColored makes from the same partons (model, not vetted)", fontsize=11)
fig.tight_layout(); plt.show()
for lab, thr, w, r, e in rows:
    print(f"  {lab:38s} p_T > {thr:2d} GeV, window {w:6s}: {r:.2f} ± {e:.2f}")

### R_AA of the chains side by side

Integrated over the comparison ranges (model statistical errors): charged jets R = 0.2 and 0.4 in STAR's
definition at 16–30 GeV (STAR's "~unbiased" range), charged hadrons 10–25 GeV (|η| < 1), pions 10–20 GeV
(|η| < 0.35), full jets R = 0.4 at 10–50 GeV; JETSCAPE's Fig. 14 and the data as the mean of their bins in
the same range.

In [ ]:
OBS = [("chjet_R0.2_star", 16, 30, "ch. jets R=0.2\n16-30 GeV"), ("chjet_R0.4_star", 16, 30, "ch. jets R=0.4\n16-30 GeV"),
       ("had", 10, 25, "h± |η|<1\n10-25 GeV"), ("pion", 10, 20, "π |η|<0.35\n10-20 GeV"), ("fjet_R0.4", 10, 50, "full jets R=0.4\n10-50 GeV")]
def band_mean(points, lo, hi, key="value"):
    v = [q[key] for q in points if q["lo"] >= lo - 1e-9 and q["hi"] <= hi + 1e-9]
    return np.mean(v) if v else np.nan
fig, ax = plt.subplots(figsize=(12, 4.6))
xs = np.arange(len(OBS))
off = np.linspace(-0.27, 0.27, len(RES) + 2)
for i, (fam, (lab, c, mk)) in enumerate([(f, FAM[f]) for f in RES]):
    vals = [integ(fam, k, lo, hi) for k, lo, hi, _ in OBS]
    ax.errorbar(xs + off[i], [v[0] for v in vals], [v[1] for v in vals], fmt=mk, ms=7, color=c, label=lab)
    print(f"{lab:52s} " + "  ".join(f"{t.split(chr(10))[0]}: {v[0]:.2f}±{v[1]:.2f}" for (k, lo, hi, t), v in zip(OBS, vals)))
jsv = [band_mean(JS14["chjet_R0.2"], 16, 30), band_mean(JS14["chjet_R0.4"], 16, 30), np.nan, band_mean(JS14["pion"], 10, 20), np.nan]
ax.plot(xs + off[-2], jsv, "s", ms=8, mfc="none", mec="C4", mew=2, label="JETSCAPE 2204.01163 Fig. 14 (Type 3, their medium)")
dat = [band_mean(STAR["0-10"]["R0.2"], 16, 30), band_mean(STAR["0-10"]["R0.4"], 16, 30), np.nan, band_mean(PHX, 10, 20), np.nan]
ax.plot(xs + off[-1], dat, "*", ms=12, color="navy", label="data: STAR jets, PHENIX π⁰")
print("JETSCAPE Fig. 14: " + "  ".join(f"{v:.2f}" for v in jsv) + " | data: " + "  ".join(f"{v:.2f}" for v in dat))
ax.set_xticks(xs); ax.set_xticklabels([o[3] for o in OBS], fontsize=9); ax.set_ylim(0, 1.05); ax.set_ylabel(r"$R_{AA}$")
ax.axhline(1, color="k", lw=0.6); ax.legend(fontsize=7.5, loc="upper center", ncol=3)
ax.set_title("Figure 7. Au+Au 0-10 %, same medium: B1, B2 (iColored), their partons with Colorless, run C; JETSCAPE and data (model, not vetted)", fontsize=10)
fig.tight_layout(); plt.show()

## Colorless's close-pair shift: most of the factor 2 (2026-11-26)

The iColored diagnosis on the GB10 (`~/prod/AuAu-RHIC_hadtests/diag/`; iColored replayed in Pythia 8.312 from
`hadronization_input/`, Colorless re-implemented on the same Pythia and checked against X-SCAPE's) found that
**Colorless, not iColored, is off in central Au+Au.**

`ColorlessHadronization.cc` ("Making collinear partons not collinear // Could have dangerous effects, yet to
be tested....") loops over all pairs of positive partons, and separately over the holes: for every pair closer
than 4 Λ_QCD = 0.8 GeV in three-momentum it adds ±0.8 GeV to p_z of the one with the larger |p_z|, and
recomputes its energy. **No energy is taken from anywhere.** The same loop is in JETSCAPE (v3.7 and every
release since v3.0, commit 959d5762, 2020), so also in the code of 2204.01163.

It fires once per close pair, so it grows with the square of the number of soft partons, i.e. the LBT
recoils and holes:

| Sample (p̂_T 23–30 GeV) | firings / event (mean) | E partons / fake ends / shift [GeV] | shift share of the hadronized E | holes: shift share |
| --- | --- | --- | --- | --- |
| vacuum `pp_ref` | 0.4 | 95 / 9.5 / 0.2 | 0.2 % | – |
| T1 (MATTER only) | 2.4 | 94 / 9.1 / 1.1 | 1.1 % | – |
| B1 (Type 2) | 751 | 185 / 17 / 337 | 62 % | 62 % |
| B2 (Type 3) | 967 | 207 / 17 / 431 | 66 % | 65 % |
| run C (all windows) | 818 | 163 / 17 / 366 | 67 % | 66 % |
| O+O `oo_med`, top 10 % N_part | 25 | 105 / 16 / 13 | 10 % | 18 % |
| O+O `oo_med`, all | 8.3 | 98 / 14 / 4.4 | 3.8 % | 8 % |

The added p_z stretches the strings along the beam: in B1 Colorless's positive hadrons carry 510–570 GeV from
≈ 183 GeV of partons, about 300 GeV of it at 2 < |η| < 4, and **35 % of the transverse momentum at |η| < 1 is
missing** (Σp_T 31 / 43 / 59 GeV against iColored's 52 / 67 / 85 GeV and the partons' 50 / 63 / 78 GeV in the
18–23 / 23–30 / 30–40 GeV windows). Without the shift Colorless gives 52 / 67 / 86 GeV, as iColored.

**The test.** X-SCAPE 299fd3f with the loop behind a switch `<JetHadronization><close_pair_shift>` (default 1;
a separate worktree and build on the GB10, the production build untouched). With the switch at 1 the patched
build reproduces the production Colorless bit for bit (B1 and run C files), and the offline re-hadronization
reproduces run C's in-job Colorless statistically. With 0, **B1's and run C's stored partons and their p+p
references** (`pp_ref`, PP19) were hadronized again by `prod_OO_isr/rehadronize_colorless.py`, with the same
seeds as the Colorless re-hadronizations above: the shift is the only difference. Hadron energy then equals the
parton energy plus the fake beam ends (≈ 17 GeV per pass) and the hadron masses.
Output: `/data/AuAu-RHIC_phys/rehad_colorless_noshift/` (648 files).

In [ ]:
SETS8 = [("auau_med", "B1, iColored"), ("auau_med_colorless", "B1's partons, Colorless (shift)"),
         ("auau_med_colorless_noshift", "B1's partons, Colorless, no shift"),
         ("js_chain_t3", "run C, Colorless (shift; = JETSCAPE)"), ("js_chain_t3_noshift", "run C's partons, Colorless, no shift"),
         ("js_chain_t3_icolored_beamends_fake", "run C's partons, iColored, beam ends, holes fake"),
         ("js_chain_t3_icolored_beamends_reconnect", "run C's partons, iColored, beam ends, holes reconnect")]
SETS8_TABLE = SETS8 + [("js_chain_t3_icolored_fake", "run C's partons, iColored, 1 GeV fakes (bad ref.)"),
                       ("js_chain_t3_icolored_reconnect", "run C's partons, iColored, reconnect + 1 GeV (bad ref.)")]
STY8 = {"auau_med": ("C3", "o", "-"), "auau_med_colorless": ("C2", "^", ":"), "auau_med_colorless_noshift": ("C2", "P", "-"),
        "js_chain_t3": ("C5", "D", ":"), "js_chain_t3_noshift": ("C5", "X", "-"),
        "js_chain_t3_icolored_beamends_fake": ("C1", "p", "-"), "js_chain_t3_icolored_beamends_reconnect": ("C6", "*", "--")}
fig, ax = plt.subplots(1, 3, figsize=(19, 4.8))
for q in PHX:
    ax[0].add_patch(plt.Rectangle((q["lo"], q["value"] - q["syst_B"]), q["hi"] - q["lo"], 2 * q["syst_B"], color="0.6", alpha=0.35, lw=0))
ax[0].errorbar([q["pt"] for q in PHX], [q["value"] for q in PHX], [q["stat"] for q in PHX], fmt="o", ms=4, color="k", label="PHENIX π⁰ 0-10 %")
for q in SH["pion"]["points"]:
    if q["hi"] > 4.5:
        ax[0].errorbar(0.5 * (q["lo"] + q["hi"]), q["value"], q["stat"], fmt="*", ms=9, color="navy")
ax[0].plot([], [], "*", ms=9, color="navy", label="STAR π± 0-12 %")
for j, R in ((1, 0.2), (2, 0.4)):
    for q in STAR["0-10"][f"R{R}"]:
        ax[j].plot(0.5 * (q["lo"] + q["hi"]), q["value"], "*", ms=10, color="navy", mfc="navy" if q["kind"] == "unbiased" else "white")
    ax[j].plot([], [], "*", ms=10, color="navy", label="STAR 0-10 % (open: biased)")
for fam, lab in SETS8:
    if fam not in RES:
        continue
    c, mk, ls = STY8[fam]
    for j, key in ((0, "pion"), (1, "chjet_R0.2_star"), (2, "chjet_R0.4_star")):
        x, xe, r, e = model(fam, key)
        ax[j].errorbar(x, r, e, xe, fmt=mk + ls, ms=5, color=c, mfc="none" if ls == ":" else c, alpha=0.6 if ls == ":" else 1, label=lab)
ax[0].set_title("pions |η| < 0.35 (model) / π⁰, π± (data)", fontsize=10); ax[0].set_xlim(4.5, 20.5); ax[0].axvspan(0, 10, color="0.92", zorder=0)
ax[1].set_title("charged jets R = 0.2, STAR's definition", fontsize=10); ax[2].set_title("charged jets R = 0.4, STAR's definition", fontsize=10)
for a_ in ax[1:]:
    a_.set_xlim(7, 32)
for a_ in ax:
    a_.axhline(1, color="k", lw=0.8); a_.set_ylim(0, 1.8); a_.set_xlabel(r"$p_T$ [GeV]"); a_.legend(fontsize=7, loc="upper left")
ax[0].set_ylabel(r"$R_{AA}$")
fig.suptitle("Figure 8. The same partons with Colorless, with and without its close-pair shift, and with iColored (model, not vetted)", fontsize=11)
fig.tight_layout(); plt.show()
OBS8 = [("pion", 8, 20), ("pion_star", 8, 20), ("had", 10, 25), ("had_star", 8, 15), ("chjet_R0.2_star", 16, 30),
        ("chjet_R0.4_star", 16, 30), ("fjet_R0.4", 10, 50)]
print(f"{'':46s}" + "".join(f"{k + ' ' + str(lo) + '-' + str(hi):>22s}" for k, lo, hi in OBS8))
for fam, lab in SETS8_TABLE:
    if fam in RES:
        print(f"{lab:46s}" + "".join(f"{'%.2f ± %.2f' % integ(fam, k, lo, hi):>22s}" for k, lo, hi in OBS8))
print("data: PHENIX pi0 8-20 " + f"{band_mean(PHX, 8, 20):.2f}" + ", STAR jets 16-30 R=0.2 / 0.4 "
      + f"{band_mean(STAR['0-10']['R0.2'], 16, 30):.2f} / {band_mean(STAR['0-10']['R0.4'], 16, 30):.2f}")
for a, b in (("js_chain_t3", "js_chain_t3_noshift"), ("auau_med_colorless", "auau_med_colorless_noshift")):
    if a in RES and b in RES:
        for k in ("had", "chjet_R0.4_star"):
            A_, B_ = RES[a]["raa"][k], RES[b]["raa"][k]
            print(f"{b} / {a}, {k}: Au+Au " + " ".join(f"{x:.1f}" for x in np.divide(B_["auau"], A_["auau"]))
                  + " | p+p " + " ".join(f"{x:.2f}" for x in np.divide(B_["pp"], A_["pp"])))

**Reading of Fig. 8** (integrated values printed above; model statistical errors ± 0.05–0.2):

- **Without the shift the Au+Au yields rise 2–9× (hadrons) and 2–11× (jets) per bin; the p+p references do not
  move** (ratio 1 within statistics: the shift hardly fires in vacuum). So R_AA rises by the same factors.
- **B1's partons with Colorless without the shift ≈ B1 with iColored** for hadrons and charged jets: charged
  hadrons 0.43 ± 0.07 vs 0.52 ± 0.08 (10–25 GeV), STAR-acceptance hadrons 0.48 vs 0.66, charged jets 0.55 / 0.64
  vs 0.51 / 0.56 (R = 0.2 / 0.4, 16–30 GeV). The hadronizations agree within ≈ 10–30 %, the remaining model
  difference the GB10 found in the per-event counts (iColored's colour-ordered strings put the leading parton
  on light strings with its recoil or colour partner; Colorless's geometric strings do not).
  **Exception: full jets R = 0.4** (10–50 GeV), 0.91 ± 0.06 without the shift against iColored's 0.51: the hole
  subtraction of the two differs for the full four-momentum in a large cone (Colorless subtracts hole *hadrons*
  made with their own fake ends; iColored's holes + partner copies at parton level). Not understood yet.
- **run C without the shift — JETSCAPE's published Type 3 with an energy-conserving Colorless — under-quenches
  like B1**: charged hadrons 0.69 ± 0.09 (10–25 GeV), STAR-acceptance hadrons 0.82 ± 0.17 (8–15 GeV), charged
  jets 0.60 ± 0.06 / 0.68 ± 0.09 (R = 0.2 / 0.4, 16–30 GeV), against PHENIX π⁰ 0.25 and STAR jets 0.30–0.31 in the
  same ranges. Its agreement with the data (and very likely Fig. 14's) rests on the shift.
- **iColored on run C's partons** (`prod_OO_isr/rehadronize_icolored.py`, GB10). Run C has no beam remnants, so
  the hard partons' beam-side colour lines are open. **First try:** iColored's own closure ends them on
  (0, 0, ±1 GeV) fake quarks; a 20 GeV parton then makes a ≈ 6 GeV string (against ≈ 30–55 GeV to a real
  remnant, or to the ±√s/6 = 33 GeV fakes of Colorless and JETSCAPE's `colored`). In vacuum (PP19's partons)
  that gives 1.1–2× the high-p_T hadrons and 1.1–1.4× the charged jets of PP19 Colorless, a reference that
  overshoots the p+p data (last two rows of the table: not usable). **Fixed:** `--beam-ends sqrts6` ends every
  beam-side line on its own quark at p_z = ±√s/6 before iColored (≈ 1.8 per Au+Au event, ≈ 4 per PP19 event);
  only the holes' lines are left to the closure. PP19 with these ends agrees with PP19 Colorless within
  5–10 % (pions 1.05–1.09, h± 1.11, charged jets 0.94, integrated 5–30 GeV): a valid reference.
- **With it, iColored on JETSCAPE's partons reproduces our chain:** holes `reconnect` (B1's closure) gives
  hadrons 0.50 ± 0.09, charged jets 0.53 / 0.46 (R = 0.2 / 0.4), full jets 0.58 — B1 gives 0.52, 0.51 / 0.56,
  0.51. Holes `fake` gives 0.40, 0.43 / 0.29, 0.37 (lower by 20–35 %: on run C's partons the holes' closure
  matters more than in B1's T2/T3 counts). The `reconnect` arm's partner copies are not subtracted (not stored
  offline, ≈ 10 GeV per event), which biases its jets up somewhat.
- **The holes' closure now matters**: per event, `fake` gives 3–23 % fewer hadrons above 8 GeV than
  `reconnect` (GB10, largest in the low windows, which dominate the cross-section-weighted spectra), and the
  `reconnect` jets are biased up by the missing partner copies. iColored's R_AA on run C's partons spans
  **0.29–0.46 for R = 0.4 jets and 0.40–0.50 for hadrons** between the two closures; with `fake` the R = 0.4
  jets reach STAR (0.29 vs 0.30), R = 0.2 jets and hadrons stay ≈ 1.4–1.6× the data.
- **So the energy-conserving hadronizations of the same partons span R_AA ≈ 0.3–0.7** (iColored 0.3–0.55
  depending on the holes' closure, Colorless without the shift 0.6–0.7), against data ≈ 0.25–0.3; Colorless
  with the shift (0.1–0.23) sits at or below the data. Run C's partons (JETSCAPE's chain, Type 3) and B1's
  (our chain, Type 2) give the same iColored R_AA with the same closure (`reconnect`): the difference between
  the chains is not the hard process or the setting. How much stronger the energy loss has to be depends on
  the hadronization and the holes' treatment: from little (iColored `fake`, R = 0.4 jets) to ≈ 2.5× in mean
  fractional loss (Colorless without the shift).
- The model's pions at 8–10 GeV are statistics-limited (grey), and run C without the shift has R_AA ≈ 1 at
  5–9 GeV: the hadron-level hole subtraction may not remove all of the recoils' hadrons at low p_T. Not checked yet.
- Run C uses our 3D medium, not JETSCAPE's 2D one; run C with the shift reproduced Fig. 14, so the media are
  equivalent to first order.

## The medium at mid-rapidity, 0–10 %, and the per-parton jet loss

The bulk check that rides along (`auau_bulk_phobos.ipynb`): charged dN/dη at |η| < 0.5 (PHOBOS 0–10 % ≈ 609,
the average of its 0–3, 3–6, 6–10 % bins at |η| < 1) and ⟨p_T⟩ of π, K, p at mid-rapidity (iSS, no
afterburner). Then the per-parton jet loss of `quench_diag.ipynb` in Au+Au, against `oo_vac` as the vacuum
reference (the jet's hadrons only, so the vacuum ratio hardly depends on the system), next to central O+O.

In [ ]:
for fam, r in RES.items():
    if "bulk_0_10" in r:
        b = r["bulk_0_10"]
        print(f"{fam}: {b['events']} events: dN_ch/deta |eta|<0.5 = {b['dnch_deta_mid']:.0f} (PHOBOS 0-10 % ≈ 609 at |eta|<1); "
              f"<pT> pi {b['mean_pt_pi']:.3f}, K {b['mean_pt_K']:.3f}, p {b['mean_pt_p']:.3f} GeV")
import glob, os
from concurrent.futures import ProcessPoolExecutor
import quench_diag as Q
D = np.load("results/quench_diag_eloss.npz")
b20 = np.quantile(D["mb_b"], 0.2)
evv = D["oo_vac/ev"].astype(int); mv0 = D["oo_vac/ev_b"][evv] < b20
evm = D["oo_med/ev"].astype(int); mm0 = D["oo_med/ev_b"][evm] < b20
PT = [10, 15, 20, 25, 30, 40, 55, 80]
def wmean(x, w):
    ok = np.isfinite(x); return (w[ok] * x[ok]).sum() / w[ok].sum()
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
xc = 0.5 * (np.array(PT[1:]) + np.array(PT[:-1]))
def vac_ref(lo, hi):
    v = mv0 & (D["oo_vac/pt"] >= lo) & (D["oo_vac/pt"] < hi)
    return wmean(D["oo_vac/r_had"][v], D["oo_vac/ev_w"][evv][v])
oo_loss = [1 - wmean(D["oo_med/r_had"][mm0 & (D["oo_med/pt"] >= lo) & (D["oo_med/pt"] < hi)],
                     D["oo_med/ev_w"][evm][mm0 & (D["oo_med/pt"] >= lo) & (D["oo_med/pt"] < hi)]) / vac_ref(lo, hi) for lo, hi in zip(PT[:-1], PT[1:])]
oo_med = [wmean((D["oo_med/e_med_hemi"] / D["oo_med/E"])[mm0 & (D["oo_med/pt"] >= lo) & (D["oo_med/pt"] < hi)],
                D["oo_med/ev_w"][evm][mm0 & (D["oo_med/pt"] >= lo) & (D["oo_med/pt"] < hi)]) for lo, hi in zip(PT[:-1], PT[1:])]
ax[0].plot(xc, 100 * np.array(oo_med), "k^--", label="O+O 0-20 % (b), current setting")
ax[1].plot(xc, 100 * np.array(oo_loss), "k^--", label="O+O 0-20 % (b), current setting")
for fam, (lab, c, mk) in FAM.items():
    if fam not in PRIMARY:                                # quench_diag needs the O+O chain's event record
        continue
    fs = sorted(f for f in glob.glob(f"/data/AuAu-RHIC_phys/{fam}_w*/*_jet_frag.h5") if os.path.exists(f.replace("_hadrons_jet_frag.h5", ".json")))
    if not fs:
        continue
    with ProcessPoolExecutor(max_workers=6) as ex:
        rr = list(ex.map(Q.eloss_file, fs))
    pt = np.concatenate([o["pt"] for *_, o in rr]); E = np.concatenate([o["E"] for *_, o in rr])
    rh = np.concatenate([o["r_had"] for *_, o in rr]); eh = np.concatenate([o["e_med_hemi"] for *_, o in rr])
    # weight: sigma / N per window (one window per file here)
    w = np.concatenate([np.full(len(o["pt"]), s[0] / max(na[0], 1)) for _, s, na, n, ev, o in rr])
    loss, med = [], []
    for lo, hi in zip(PT[:-1], PT[1:]):
        m = (pt >= lo) & (pt < hi)
        loss.append(1 - wmean(rh[m], w[m]) / vac_ref(lo, hi) if m.sum() > 30 else np.nan)
        med.append(wmean((eh / E)[m], w[m]) if m.sum() > 30 else np.nan)
    ax[0].plot(xc, 100 * np.array(med), mk + "-", color=c, label=f"Au+Au 0-10 %, {lab}")
    ax[1].plot(xc, 100 * np.array(loss), mk + "-", color=c, label=f"Au+Au 0-10 %, {lab}")
    print(f"{fam}: {len(pt)} partons; energy to the medium [%] " + " ".join(f"{x:.1f}" for x in 100 * np.array(med))
          + " | jet loss (hadron) [%] " + " ".join(f"{x:.1f}" for x in 100 * np.array(loss)))
ax[0].set_title("energy given to the medium, hemisphere [% of the parton's E]", fontsize=9)
ax[1].set_title("jet loss 1 - <r>_med/<r>_vac (hadron level, R = 0.4), vacuum: oo_vac", fontsize=9)
for a_ in ax:
    a_.set_xlabel("parton $p_T$ [GeV]"); a_.legend(fontsize=7); a_.set_ylim(0, None)
fig.suptitle("Figure 3. Per-parton energy loss: central Au+Au against central O+O (model, not vetted)", fontsize=11)
fig.tight_layout(); plt.show()

## Reading (2026-11-26: B1, B2, run C, the Colorless re-hadronizations and T1–T3 complete)

B1 = the current O+O setting (Matter type 1 = 2204.01163's Type 2, α_s 0.25, Lbt α_s 0.25, Q0 2, checked
in the job logs), 19,999 Au+Au events at b < 4.7 fm, all 10 windows; p+p: 66k `pp_ref` events in the cut.
Averages below are inverse-variance weighted, model statistical errors only. **Model results, not vetted.**

> **Correction (2026-11-26, late; point 13 and Fig. 8).** Points 11–12 conclude that iColored makes the factor 2.
> That is superseded: **most of it is Colorless's close-pair shift**, which does not conserve energy and removes
> ≈ 35 % of the mid-rapidity p_T in central Au+Au. Without it Colorless ≈ iColored (within 10–30 %), and
> JETSCAPE's Type 3 (run C) under-quenches like B1. Points 11–12 are kept as written, for the record.

1. **B1 under-quenches central Au+Au: R_AA ≈ 1.3–2× the data, for hadrons and jets.**
   - Hadrons, 10–15 GeV (integrated; the model is statistics-limited below 10 GeV): π± at |y| < 0.5
     0.45 ± 0.10 against STAR π± 0.31–0.32 (0–12 %) and PHENIX π⁰ 0.23–0.24 (10–14 GeV). Model/data
     ≈ 1.4 (STAR π±) to ≈ 1.9 (PHENIX π⁰), each ± ≈ 0.3 from the model's statistics.
   - STAR's charged hadrons stop at 10 GeV, where the model becomes usable, so they do not test it directly:
     the model's 0.37 ± 0.07 at 10–15 GeV sits above STAR's last points (8–10 GeV: 0.19–0.22 at 0–5 %,
     0.25–0.32 at 10–20 %), with R_AA nearly flat in p_T in the data.
   - STAR's π± lie ≈ 0.07 (≈ 30 %) above PHENIX's π⁰; that is within their combined systematic and
     normalisation uncertainties (STAR ± 0.04–0.05 and 12 %, PHENIX ± 0.03 and 9.7 %).
   - Charged jets with STAR's definition (the 5 GeV leading-track cut on Au+Au only), in STAR's
     "~unbiased" range 16–30 GeV: 0.46–0.57 for all three R. STAR: 0.25–0.38. Model/STAR ≈ 2.1–2.3 at
     16–20 GeV, 1.2–1.5 at 20–30 GeV, beyond STAR's correlated + shape systematics.
   - Hadrons and jets are both off by about the same factor: the overall strength is too low, not the shape
     of the jet or the cone.
2. **The leading-track cut matters in the biased region, and only there.**
   - STAR's PYTHIA reference has no cut, so at low jet p_T the cut removes p+p jets too and lowers R_AA.
     With STAR's definition the model drops at 8–14 GeV by 15–55 % (R = 0.4, 8–10 GeV: 0.87 with the cut in
     both → 0.38), and now rises with p_T (R = 0.4: 0.38 → ≈ 0.55), as STAR does. It is still 2–3× STAR there.
   - Above 15–17 GeV the definitions agree to ≤ 4 %.
   - Our cut reproduces STAR's PYTHIA bias: in `pp_ref` the fraction of jets keeping a 5 GeV leading hadron
     passes 0.9 at 12–14 GeV (R = 0.2) and 16–20 GeV (R = 0.4); STAR's Fig. 13: 13 and 17 GeV.
   - The model's R dependence in the unbiased range is weak, like STAR's; B2 is flat too (0.50 at R = 0.2 and 0.4).
3. **Medium response is large in central Au+Au.** Without the hole + partner subtraction, R = 0.4 charged
   jets are 1.2–1.36× higher and full R = 0.4 jets go from 0.51 to 0.77 (Fig. 2, dotted): about a third of
   the energy lost from an R = 0.4 cone comes back as recoils. In O+O this is negligible. The R = 0.2 jets
   change by only 3–9 %.
4. **The model's hadron R_AA below 10 GeV is limited by statistics, not physics.** At 5–10 GeV it scatters
   around 1 with ± 0.3–0.9 per bin. Those hadrons come mostly from the lowest p̂_T windows (5–11 GeV), where only
   ≈ 10–50 of the 2000 Au+Au events per window carry one (a hadron near the parton's p_T is rare); in Au+Au the
   5–7 GeV window alone holds half of the 5–8 GeV yield, from 10 events. Medium response (hadrons are not
   hole-subtracted) may add at the low end, but cannot be separated with these statistics. Read hadrons from
   10 GeV on. A test of hadron R_AA below 10 GeV needs many more events at p̂_T 5–15 GeV (cheap: a hadron-level
   p+p and Au+Au sample with more events in the low windows), not more of the high windows.
5. **Per parton (Fig. 3), Au+Au is much stronger than O+O, but still not enough.** A 30–40 GeV parton
   gives 12 % of its energy to the medium (O+O 0–20 %: 5 %), and its R = 0.4 jet loses 18 % at hadron level
   (O+O: 2.4 %). At 10–15 GeV: 27 % to the medium, 37 % jet loss. The loss falls steeply with p_T.
   The average loss and R_AA are not the same thing: event-by-event fluctuations (short paths, partons
   near the surface) keep R_AA higher than a mean-loss estimate would.
6. **The bulk is fine for this test.** dN_ch/dη at |η| < 0.5 = 628 (PHOBOS ≈ 609); ⟨p_T⟩ π / K / p
   = 0.48 / 0.72 / 1.04 GeV, a few % harder than the measured central values. Neither causes a factor 2.
7. **Consistent with O+O.** There the model is above sPHENIX by 1–2σ per bin. Au+Au says the same thing
   with much more weight: the current setting loses too little energy.

8. **The p+p reference: a ± 15–25 % systematic on the jet comparison, none on the hadrons** (Figs. 4–5).
   - `pp_ref` describes the measured p+p hadrons as well as Pythia 8 does (weighted model/data): STAR π±
     6–15 GeV 1.12 ± 0.06, PHENIX π⁰ 6–20 GeV 0.86 ± 0.05, STAR h± 6–10 GeV 0.88 ± 0.06; Pythia 8 with the same
     decays: Monash 1.07 / 1.00 / 0.90, Detroit 1.00 / 0.87 / 0.74. The hadron R_AA data use these measured
     references, so the hadron gap is not a reference effect.
   - Charged jets (UE-subtracted, the same decays): `pp_ref` / Detroit 1.01–1.15 (R = 0.2), 0.84–0.96
     (R = 0.4); / Monash 0.83–1.01 (R = 0.2), 0.69–0.84 (R = 0.4). Our jets are somewhat narrower than
     Pythia 8's. Against STAR's measured p+p full jets (R = 0.5) `pp_ref` is right (tuned there) and Pythia 8 is
     10–40 % above (tuning page; full jets hardly depend on the decays).
   - Divided by Pythia 8 as STAR does, the model's R_AA at 16–30 GeV is, for R = 0.2: 0.49–0.58 (Detroit),
     0.40–0.43 (Monash) against 0.46–0.52 with `pp_ref`; for R = 0.4: 0.41–0.57 (Detroit), 0.36–0.44 (Monash)
     against 0.47–0.61. STAR: 0.25–0.38. The reference moves the model by + 10 % to − 25 %, in both
     directions; how STAR's PYTHIA 6 compares with real p+p jets we cannot check here.
   - An earlier version of this point used Pythia 8 with its own decays (weak decays on, π⁰ and η stable), which
     overstated Pythia's charged jets by 5–18 % and the reference effect with them.
   - JETSCAPE's PP19 (their p+p baseline) is within ≈ 10 % of `pp_ref` too (point 9).
   - What is left: jets 1.3–2× STAR at any of the references, hadrons 1.4–1.9× the data. B1 still under-quenches.

9. **JETSCAPE's own chain with the published default reaches the data; ours with Type 2 is about 2× above
   it** (purple steps in Figs. 1–2; 2204.01163 Fig. 14, no retuning at RHIC).
   - JETSCAPE Type 3: charged jets 0.22–0.31 (R = 0.2), 0.19–0.29 (R = 0.4) at 8–30 GeV; pions 0.18–0.28 at
     8–20 GeV. That is STAR's 0.25–0.36 (unbiased) and PHENIX's 0.19–0.38 within ≈ 10–20 %: a little above
     STAR in the biased low-p_T bins (R = 0.4, 8–10 GeV: 0.19 against 0.12), a little below at 25–30 GeV
     (0.28 against 0.36).
   - B1 (our chain, Type 2 α_s 0.25): jets 0.46–0.61, pions ≈ 0.45, i.e. ≈ 2× JETSCAPE's curve at the same
     observables. In Pb+Pb at 5.02 TeV the paper finds Type 2 (0.25) and Type 3 (0.3) equal for jets and Type 2
     *more* suppressing for high-p_T hadrons, so the q̂ form alone is not expected to make a factor 2.
   - B2 (Type 3 α_s 0.3 in our chain) is the direct test: if it lands near JETSCAPE's curve, the setting was the
     cause; if it stays near B1, the difference is in the chain (medium, start time, iMATTER ISR, iColored,
     the p+p baseline).
   - The p+p baseline is not it: JETSCAPE's PP19 (same decays, 160k events) is within ≈ 10 % of `pp_ref`
     (UE-subtracted charged jets pp_ref / PP19 0.85–1.08 at R = 0.2, 0.77–0.94 at R = 0.4; hadrons against data
     PP19 1.03 / 0.96 / 0.82 vs pp_ref 1.12 / 0.86 / 0.88). With PP19 as the reference B1's jet R_AA is 0.42–0.55
     (R = 0.2) and 0.33–0.53 (R = 0.4): still ≈ 1.5–2× JETSCAPE's curve.

10. **Run C: JETSCAPE's jet chain on our medium reproduces JETSCAPE's Fig. 14 (2026-11-26).** Charged jets
    (STAR's definition, 16–30 GeV) 0.23 ± 0.03 (R = 0.2) and 0.17 ± 0.02 (R = 0.4), hadrons 0.20 ± 0.03
    (10–25 GeV), pions 0.19 ± 0.07 (10–20 GeV), full jets 0.19 (10–50 GeV), against JETSCAPE's 0.22–0.31 and the
    data (STAR 0.25–0.36, PHENIX 0.23–0.38): at or slightly below both. **The medium is not the culprit**: our
    string-sourced 3D medium quenches at least as much as TRENTo + free streaming + VISHNU.
11. **The hadronization is (Figs. 6–7).** On the same medium:
    - The energy given to the medium (recoils − holes) is about the same in B1, B2 and run C (12–18 GeV per
      event), and so are the holes (60–100 per event) and the showers above 5 GeV; at parton level B1 is not
      less quenched than run C (final partons above 15 GeV in 23–30 GeV: 0.31 vs 0.39 per event, the same
      shower scheme).
    - **B1's own partons, hadronized with Colorless** (`prod_OO_isr/rehadronize_colorless.py`, same events),
      give run C's R_AA: charged jets 0.23 ± 0.02 (R = 0.2) and 0.21 ± 0.02 (R = 0.4), full jets 0.22, hadrons
      0.15 ± 0.03 — down from 0.51 / 0.56 / 0.51 / 0.52 with iColored.
    - High-p_T hadrons per event from the same partons (23–30 GeV window, charged, p_T > 12 GeV): Au+Au
      iColored 0.104, Colorless 0.052 (2.0×); in p+p (vacuum) 0.152 vs 0.146 (1.04×). In 40–55 GeV: 1.6× vs
      1.1×. **iColored in the medium makes about twice the high-p_T hadrons of Colorless from the same
      partons; in vacuum the two agree.** The likely mechanism: `colour_closure reconnect` joins the recoils
      and the holes' colour partners to the jet partons, and string fragmentation recaptures the energy the
      partons gave to the recoils into leading hadrons and jets. Colorless hadronizes without that colour
      flow (and is the hadronization 2204.01163 was tuned with).
    - **The q̂ setting barely matters with iColored: B2** (Type 3, α_s 0.3, the published default; complete,
      19,999 events) gives charged jets 0.50 ± 0.05 (R = 0.2 and 0.4, 16–30 GeV), full jets 0.50, hadrons
      0.59 ± 0.09 (10–25 GeV), pions 0.71 ± 0.18 (10–20 GeV): like B1 (0.51 / 0.56, 0.51, 0.52, 0.45), and 0.098
      high-p_T hadrons per event in 23–30 GeV (B1 0.104). With iColored, the published setting under-quenches as
      much as B1; with Colorless it is run C (0.17–0.23).
    - Caveat, the shower scheme: the Au+Au medium arm showers to QS = 0.9 GeV (A+A scheme), `pp_ref` to Q0 =
      2 GeV. Measured in O+O events without a medium (`oo_analysis_rhic.ipynb` 6b), this mismatch lowers the
      Colorless R_AA by ≈ 7 % (jets) and ≈ 18 % (hadrons) but iColored's by only ≈ 5 % and 4–10 %: B1's partons
      with Colorless would be ≈ 0.25 (charged jets) and ≈ 0.18 (hadrons) against a reference in the same scheme,
      still ≈ 2× below iColored. Run C and JETSCAPE (PP19 to Q0 = 1 GeV) are nearly scheme-consistent.
    - **T1** (GB10: B1's job with `--matter-only`, i.e. no LBT, recoils, holes or closure; windows 18–23 /
      23–30 / 30–40 GeV, 500 events each; Fig. 6, dashed): iColored / Colorless on the same partons, charged
      hadrons > 8 | > 12 GeV: 1.78 | 1.54, 1.20 | 1.07, 1.42 | 1.42 (typically ≈ 1.4); B1 2.63 | 3.72, 1.97 |
      1.98, 1.71 | 2.20 (≈ 2.2); vacuum 1.04–1.10. **Two parts:** iColored hadronizes a medium-modified MATTER
      shower ≈ 1.4× harder than Colorless even without recoils, holes or closure; LBT with its recoils, holes and
      the closure add about another 1.5×.
    - **T2 / T3: the closure does not matter** (B1 with `--colour-closure thermal`, GB10, / `fake`, 3090s; same
      seeds, same energy loss: 14.5 GeV to the medium and 79 holes per event in 23–30 GeV, as B1). iColored /
      Colorless, hadrons > 8 GeV, 18–23 / 23–30 / 30–40 GeV: thermal 2.10 / 1.96 / 1.58, fake 2.32 / 1.92 / 1.47,
      reconnect (B1) 2.63 / 1.97 / 1.71 (errors ≈ 0.1–0.5). Joining the recoils to the jet's strings (`reconnect`),
      closing each hole on its own medium partner (`thermal`) or on soft beam partons (`fake`): the same factor.
      **The excess is iColored's string fragmentation of a medium-modified shower** (≈ 1.4× from the MATTER
      modification alone, ≈ 2× with LBT's recoils and holes), **not the way the open colour lines are closed.** The duplicate-tag fix changed iColored by 5–17 % and the closures with Matter
      recoils by ≲ 1 % (`prod_OO_isr/hadro_compare`, O+O, no LBT): neither makes a factor 2.
    - Caveat on Colorless with this chain: PythiaIsrGun's hard partons and their Matter daughters carry status
      1000, which Colorless drops (`prod_OO_isr/README.md`, "Not working yet"); the re-hadronization relabels
      them 0 (in Au+Au ≈ 2 partons, 14 GeV per event; in vacuum `pp_ref` most of the jet).
12. **Consequences.**
    - The energy-loss parameters of 2204.01163 are calibrated with Colorless; with iColored they under-quench by
      ≈ 2× in R_AA (B2). The closure of the holes' colour lines does not help (T2, T3): with iColored the coupling
      would have to be retuned, or the medium-modified showers hadronized with Colorless (or a validated hybrid).
    - With Colorless, Type 2 at α_s 0.25 (B1's partons) gives hadrons 0.15 and pions 0.07: over-suppressed, as
      the paper finds for Type 2 at the LHC; Type 3 is closer to the data.
    - **B2's partons with Colorless** (Type 3 in our full chain: iMATTER ISR, our hard tune): charged jets
      0.27 ± 0.03 (R = 0.2) and 0.23 ± 0.03 (R = 0.4), full jets 0.22, hadrons 0.19 ± 0.03, pions 0.24 ± 0.09 —
      the same as run C (0.23, 0.17, 0.19, 0.20, 0.19): with the same hadronization and setting, our hard process and
      ISR differ from JETSCAPE's chain by ≲ 10–30 %. The whole factor 2 is the hadronization.
    - The O+O results (phase 3, iColored) carry the same effect: their small quenching against sPHENIX is in part
      this hadronization. *(Superseded by 13.)*
13. **Colorless's close-pair shift, not iColored, makes most of the factor 2 (Fig. 8, 2026-11-26).**
    - Colorless pushes every pair of partons closer than 0.8 GeV apart along the beam without taking the energy
      from anywhere. With ≈ 100 soft recoils and holes per central Au+Au event, ≈ 2/3 of the energy Colorless
      hadronizes comes from this shift (B1, B2, run C alike), and ≈ 35 % of the mid-rapidity p_T is lost. In
      vacuum and with MATTER only it hardly fires (< 1 %), which is why the vacuum checks agreed.
    - Without the shift, the same partons give: B1 with Colorless ≈ B1 with iColored (hadrons and charged jets
      within 10–30 %; full jets differ, see Fig. 8); run C (JETSCAPE's chain and published Type 3) charged
      hadrons 0.69, charged jets 0.60–0.68, against data 0.25–0.31 in the same ranges.
    - iColored on run C's partons (remnant-like √s/6 beam ends, holes `reconnect`) gives hadrons 0.50 and charged
      jets 0.46–0.53, as B1 (0.52, 0.51–0.56): JETSCAPE's Type 3 partons hadronized like ours give our R_AA. With
      the holes closed by `fake` it is 20–35 % lower (R = 0.4 jets 0.29, at STAR): the holes' colour treatment is
      now the largest open systematic of iColored in Au+Au.
    - So iColored conserves energy and is close to an energy-conserving Colorless; the remaining 10–30 %
      (T1: up to 1.65 at 18–23 GeV) is a real model difference (string topology). **The energy loss — Type 2 and
      Type 3 alike — is too weak for an energy-conserving hadronization**; the published Type 3 calibration
      (Colorless since JETSCAPE v3.0) absorbed the shift. How much stronger depends on the hadronization and the
      holes' closure (Fig. 8).
    - O+O: the shift is small there (≈ 10 % of the hadronized energy in central events, ≈ 4 % on average), and our
      O+O chain uses iColored: the O+O under-quenching points to the energy loss, not the hadronization.

**What next:**
- **The holes' closure on run C's partons** (`fake` vs `reconnect`: 20–35 % in R_AA, Fig. 8) and the
  partner-copy subtraction offline (`reconnect`).
- **Recalibrate the energy loss with an energy-conserving hadronization** (iColored, or Colorless without the
  shift), in Au+Au first: a q̂ / α_s scan with `run_prod_jet_single.py` and hydro reuse is cheap. Then O+O.
- **Colorless's hole subtraction without the shift**: full jets R = 0.4 (0.91 vs iColored's 0.51) and the
  low-p_T hadrons (run C ≈ 1 at 5–9 GeV).
- **Report the shift to JETSCAPE / X-SCAPE** (the patch is a switch, default unchanged); decide whether X-SCAPE
  changes the default.
- **A p+p reference in the A+A scheme** (phase 4: Matter `in_vac 0`), which removes the shower-scheme term from
  every comparison.
- The O+O consequences are in `oo_analysis_rhic.ipynb`, section 6b (to be updated with point 13).